# 08_3_6 DQA Scene Phase2 SCoLQ Policy

`01_train_and_select_scolq.ipynb` の結果を Phase2 に入れる実験。SCoLQ = **Source-Calibrated Localization Quality Judge** を使い、teacher confidence ではなく source GT で校正した bbox localization quality を pseudo label の gate score にする。

今回の読みはかなり明確で、生 pseudo box は source_val で `good50=9.1%` しかない一方、SCoLQ 上位は `top 5% good50=87.9%`, `top 10% good50=65.8%`, `top 20% good50=40.7%` まで濃縮できていた。なので 08_3_6 では、bbox regression を SCoLQ 高スコア box に限定し、低スコア box は objectness の弱い教師としてだけ残す。

注意点として、`rider/bus/bike/motor/train` は class 別診断で弱い。variant C ではこれらの class の bbox gate を高くして、rare/weak class の localization drift を避ける。

In [ ]:
from __future__ import annotations

import json
import os
import signal
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "dynamic_quality_aware_classwise_aggregation").exists() and (path / "navigating_data_heterogeneity").exists():
            return path
    raise RuntimeError(f"Could not find repo root from {start}")


REPO_ROOT = find_repo_root(Path.cwd())
DQA_ROOT = REPO_ROOT / "dynamic_quality_aware_classwise_aggregation"
RUN_SCRIPT = DQA_ROOT / "run_dqa_cwa_fedsto_scene_v2_phase2_scolq_policy.py"
EVAL_SCRIPT = DQA_ROOT / "evaluate_scene_protocol.py"
SOURCE_WORK_ROOT = DQA_ROOT / "efficientteacher_dqa08_scene_tri_stage_policy_8h"
SCOLQ_ROOT = DQA_ROOT / "source_calibrated_localization_quality"
SCOLQ_MODEL = SCOLQ_ROOT / "artifacts" / "scolq_best.joblib"
SCOLQ_RANKING = SCOLQ_ROOT / "reports" / "model_ranking.csv"
SCOLQ_DIAG = SCOLQ_ROOT / "reports" / "best_model_class_diagnostics.csv"
BASE_WORK_ROOT = DQA_ROOT / "efficientteacher_dqa08_3_6_phase2_scolq_policy"
BASE_STATS_ROOT = DQA_ROOT / "stats_dqa08_3_6_phase2_scolq_policy"
BASE_LOG_ROOT = DQA_ROOT / "logs_dqa08_3_6_phase2_scolq_policy"

for path in [RUN_SCRIPT, EVAL_SCRIPT, SOURCE_WORK_ROOT, SCOLQ_MODEL]:
    print(path, "exists=", path.exists())

BASE_WORK_ROOT.mkdir(parents=True, exist_ok=True)
BASE_STATS_ROOT.mkdir(parents=True, exist_ok=True)
BASE_LOG_ROOT.mkdir(parents=True, exist_ok=True)

if SCOLQ_RANKING.exists():
    ranking = pd.read_csv(SCOLQ_RANKING)
    display(ranking.head(8)[["candidate", "n_features", "ap50_quality", "ap75_quality", "p50_at_10pct", "p50_at_20pct", "selection_score"]])
if SCOLQ_DIAG.exists():
    diag = pd.read_csv(SCOLQ_DIAG)
    display(diag[["class_name", "base_good50_rate", "p50_at_10pct", "p50_at_20pct"]])

print("workspace:", BASE_WORK_ROOT)
print("stats:", BASE_STATS_ROOT)
print("logs:", BASE_LOG_ROOT)

## Variant Design

SCoLQ の source_val threshold sweep では `scolq>=0.60` が coverage `5.5%` で `good50=85.2%`、`scolq>=0.70` が coverage `4.6%` で `good50=89.7%` だった。bbox regression に流すなら `0.60-0.70` が現実的。

- A: まずの本命。`high=0.60` で SCoLQ 高品質 box だけ bbox/cls positive、残りは弱 objectness。
- B: precision 優先。`high=0.70`、pseudo 数も絞る。
- C: class-wise guard。SCoLQ 診断で弱かった `rider/bus/bike/motor/train` の bbox gate を上げる。
- D: bbox をほぼ使わず、SCoLQ を DQA quality と objectness だけに使う対照。

In [ ]:
PHASE2_ROUNDS_PER_VARIANT = 10
BATCH_SIZE = 160
WORKERS = 10
GPUS = 2
MASTER_PORT_BASE = 29960
STREAM_TRAIN_OUTPUT = False
MIN_FREE_GIB = 30

# まずは本命Aだけ回す。比較を増やすなら [] にすると全variant、または C/B/D を追加する。
SELECTED_VARIANTS: list[str] = ["a_scolq_soft_bbox_r003"]

CLASSWISE_WEAK_GUARD_HIGH = [0.60, 0.80, 0.55, 0.80, 0.65, 0.80, 0.80, 0.60, 0.60, 0.90]

VARIANTS = [
    {
        "name": "a_scolq_soft_bbox_r003",
        "description": "SCoLQ>=0.60だけをbbox/cls positiveにする本命。低SCoLQは弱objectnessとして残す。",
        "source_phase1_round": 3,
        "dqa_start_phase": 2,
        "client_train_scope": "all",
        "classwise_blend": 0.065,
        "server_anchor": 13.0,
        "temperature": 2.7,
        "stability_lambda": 0.66,
        "residual_start": 0.14,
        "residual_end": 0.06,
        "min_server_alpha_start": 0.70,
        "min_server_alpha_end": 0.74,
        "env": {
            "DQA0836_SCOLQ_LOW": 0.10,
            "DQA0836_SCOLQ_MID": 0.30,
            "DQA0836_SCOLQ_HIGH": 0.60,
            "DQA0836_NMS_CONF_THRES": 0.01,
            "DQA0836_TEACHER_LOSS_WEIGHT": 0.32,
            "DQA0836_BOX_LOSS_WEIGHT": 0.010,
            "DQA0836_OBJ_LOSS_WEIGHT": 0.32,
            "DQA0836_CLS_LOSS_WEIGHT": 0.08,
            "DQA0836_LOW_MID_OBJ_WEIGHT": 0.25,
            "DQA0836_MID_HIGH_OBJ_WEIGHT": 0.70,
            "DQA0836_CLIENT_LR0_START": 0.0010,
            "DQA0836_CLIENT_LR0_END": 0.00022,
            "DQA0836_SERVER_LR0_START": 0.0030,
            "DQA0836_SERVER_LR0_END": 0.0010,
            "DQA0836_MAX_PSEUDO_PER_IMAGE": 80,
            "DQA0836_MAX_PSEUDO_PER_CLASS_IMAGE": 25,
        },
    },
    {
        "name": "b_scolq_strict_bbox_r003",
        "description": "SCoLQ>=0.70だけbbox positive。bboxはかなり高精度に寄せ、pseudo数も絞る。",
        "source_phase1_round": 3,
        "dqa_start_phase": 2,
        "client_train_scope": "all",
        "classwise_blend": 0.050,
        "server_anchor": 16.0,
        "temperature": 3.0,
        "stability_lambda": 0.72,
        "residual_start": 0.10,
        "residual_end": 0.03,
        "min_server_alpha_start": 0.76,
        "min_server_alpha_end": 0.84,
        "env": {
            "DQA0836_SCOLQ_LOW": 0.15,
            "DQA0836_SCOLQ_MID": 0.40,
            "DQA0836_SCOLQ_HIGH": 0.70,
            "DQA0836_NMS_CONF_THRES": 0.015,
            "DQA0836_TEACHER_LOSS_WEIGHT": 0.28,
            "DQA0836_BOX_LOSS_WEIGHT": 0.008,
            "DQA0836_OBJ_LOSS_WEIGHT": 0.28,
            "DQA0836_CLS_LOSS_WEIGHT": 0.05,
            "DQA0836_LOW_MID_OBJ_WEIGHT": 0.18,
            "DQA0836_MID_HIGH_OBJ_WEIGHT": 0.55,
            "DQA0836_CLIENT_LR0_START": 0.0008,
            "DQA0836_CLIENT_LR0_END": 0.00016,
            "DQA0836_SERVER_LR0_START": 0.0030,
            "DQA0836_SERVER_LR0_END": 0.0009,
            "DQA0836_MAX_PSEUDO_PER_IMAGE": 55,
            "DQA0836_MAX_PSEUDO_PER_CLASS_IMAGE": 16,
        },
    },
    {
        "name": "c_scolq_classwise_weak_guard_r003",
        "description": "弱いclassのbbox gateを上げる。rare classの誤bbox regressionで全体が崩れるのを避ける。",
        "source_phase1_round": 3,
        "dqa_start_phase": 2,
        "client_train_scope": "neck_head",
        "classwise_blend": 0.060,
        "server_anchor": 14.0,
        "temperature": 2.8,
        "stability_lambda": 0.68,
        "residual_start": 0.12,
        "residual_end": 0.04,
        "min_server_alpha_start": 0.74,
        "min_server_alpha_end": 0.82,
        "env": {
            "DQA0836_SCOLQ_LOW": 0.10,
            "DQA0836_SCOLQ_MID": 0.30,
            "DQA0836_CLASSWISE_HIGH": json.dumps(CLASSWISE_WEAK_GUARD_HIGH),
            "DQA0836_NMS_CONF_THRES": 0.01,
            "DQA0836_TEACHER_LOSS_WEIGHT": 0.30,
            "DQA0836_BOX_LOSS_WEIGHT": 0.010,
            "DQA0836_OBJ_LOSS_WEIGHT": 0.32,
            "DQA0836_CLS_LOSS_WEIGHT": 0.06,
            "DQA0836_LOW_MID_OBJ_WEIGHT": 0.22,
            "DQA0836_MID_HIGH_OBJ_WEIGHT": 0.65,
            "DQA0836_CLIENT_LR0_START": 0.0009,
            "DQA0836_CLIENT_LR0_END": 0.00018,
            "DQA0836_SERVER_LR0_START": 0.0030,
            "DQA0836_SERVER_LR0_END": 0.0010,
            "DQA0836_MAX_PSEUDO_PER_IMAGE": 80,
            "DQA0836_MAX_PSEUDO_PER_CLASS_IMAGE": 20,
        },
    },
    {
        "name": "d_scolq_obj_only_dqa_r003",
        "description": "bbox regressionをほぼ切る対照。SCoLQをDQA qualityと弱objectnessだけに使う。",
        "source_phase1_round": 3,
        "dqa_start_phase": 2,
        "client_train_scope": "neck_head",
        "classwise_blend": 0.055,
        "server_anchor": 15.0,
        "temperature": 2.8,
        "stability_lambda": 0.70,
        "residual_start": 0.08,
        "residual_end": 0.02,
        "min_server_alpha_start": 0.78,
        "min_server_alpha_end": 0.88,
        "env": {
            "DQA0836_SCOLQ_LOW": 0.10,
            "DQA0836_SCOLQ_MID": 0.35,
            "DQA0836_SCOLQ_HIGH": 0.95,
            "DQA0836_NMS_CONF_THRES": 0.01,
            "DQA0836_TEACHER_LOSS_WEIGHT": 0.30,
            "DQA0836_BOX_LOSS_WEIGHT": 0.000,
            "DQA0836_OBJ_LOSS_WEIGHT": 0.34,
            "DQA0836_CLS_LOSS_WEIGHT": 0.00,
            "DQA0836_LOW_MID_OBJ_WEIGHT": 0.25,
            "DQA0836_MID_HIGH_OBJ_WEIGHT": 0.60,
            "DQA0836_CLIENT_LR0_START": 0.0008,
            "DQA0836_CLIENT_LR0_END": 0.00016,
            "DQA0836_SERVER_LR0_START": 0.0030,
            "DQA0836_SERVER_LR0_END": 0.0010,
            "DQA0836_MAX_PSEUDO_PER_IMAGE": 80,
            "DQA0836_MAX_PSEUDO_PER_CLASS_IMAGE": 25,
        },
    },
]

selected = [v for v in VARIANTS if not SELECTED_VARIANTS or v["name"] in SELECTED_VARIANTS]
print("selected:", [v["name"] for v in selected])

In [ ]:
def variant_work_root(variant: dict) -> Path:
    return BASE_WORK_ROOT / variant["name"]


def variant_stats_root(variant: dict) -> Path:
    return BASE_STATS_ROOT / variant["name"]


def variant_runner_log(variant: dict) -> Path:
    return BASE_LOG_ROOT / f"{variant['name']}_runner.out"


def variant_train_log(variant: dict) -> Path:
    return BASE_LOG_ROOT / f"{variant['name']}_train.log"


def variant_pid_path(variant: dict) -> Path:
    return BASE_LOG_ROOT / f"{variant['name']}.pid"


def variant_env(variant: dict) -> dict[str, str]:
    env = os.environ.copy()
    stats_root = variant_stats_root(variant)
    stats_root.mkdir(parents=True, exist_ok=True)
    env["DQA0836_VARIANT"] = variant["name"]
    env["DQA0836_SOURCE_WORK_ROOT"] = str(SOURCE_WORK_ROOT)
    env["DQA0836_SOURCE_PHASE1_ROUND"] = str(variant["source_phase1_round"])
    env["DQA0836_SCOLQ_MODEL"] = str(SCOLQ_MODEL)
    env["DQA0836_CLIENT_TRAIN_SCOPE"] = variant["client_train_scope"]
    env["DQA0836_SERVER_TRAIN_SCOPE"] = "all"
    env["DQA0836_RESIDUAL_START"] = str(variant["residual_start"])
    env["DQA0836_RESIDUAL_END"] = str(variant["residual_end"])
    env["DQA0836_MIN_SERVER_ALPHA_START"] = str(variant["min_server_alpha_start"])
    env["DQA0836_MIN_SERVER_ALPHA_END"] = str(variant["min_server_alpha_end"])
    env["DQA0836_PHASE2_ROUNDS"] = str(PHASE2_ROUNDS_PER_VARIANT)
    env["DQA08_STATS_ROOT"] = str(stats_root.resolve())
    env["DQA08_THRESHOLD_LOG"] = str((stats_root / "phase2_scolq_policy_schedule.jsonl").resolve())
    env["DQA_STATS_QUALITY_MODE"] = "scolq"
    env["DQA0834_STATS_QUALITY_MODE"] = "scolq"
    env.setdefault("ET_SKIP_AFTER_TRAIN_BEST_VAL", "1")
    for key, value in variant.get("env", {}).items():
        env[key] = str(value)
    return env


def train_cmd(variant: dict, *, stream: bool = STREAM_TRAIN_OUTPUT) -> list[str]:
    cmd = [
        sys.executable,
        str(RUN_SCRIPT),
        "--workspace-root",
        str(variant_work_root(variant)),
        "--stats-root",
        str(variant_stats_root(variant)),
        "--phase1-rounds",
        "0",
        "--phase2-rounds",
        str(PHASE2_ROUNDS_PER_VARIANT),
        "--batch-size",
        str(BATCH_SIZE),
        "--workers",
        str(WORKERS),
        "--gpus",
        str(GPUS),
        "--master-port",
        str(MASTER_PORT_BASE + selected.index(variant)),
        "--log-file",
        str(variant_train_log(variant)),
        "--classwise-blend",
        str(variant["classwise_blend"]),
        "--server-anchor",
        str(variant["server_anchor"]),
        "--temperature",
        str(variant["temperature"]),
        "--stability-lambda",
        str(variant["stability_lambda"]),
        "--dqa-start-phase",
        str(variant["dqa_start_phase"]),
        "--min-free-gib",
        str(MIN_FREE_GIB),
    ]
    if stream:
        cmd.append("--stream-train-output")
    return cmd


def history_rows(variant: dict) -> list[dict]:
    path = variant_work_root(variant) / "history.json"
    if not path.exists():
        return []
    return json.loads(path.read_text())


def read_pid(path: Path) -> int | None:
    if not path.exists():
        return None
    try:
        return int(path.read_text().strip())
    except Exception:
        return None


def pid_state(pid: int | None) -> str:
    if pid is None:
        return "no pid"
    try:
        os.kill(pid, 0)
    except ProcessLookupError:
        return "stopped"
    except PermissionError:
        return "running?"
    return "running"


def tail_lines(path: Path, n: int = 30) -> list[str]:
    if not path.exists():
        return [f"missing: {path}"]
    return path.read_text(errors="replace").splitlines()[-n:]

print("helpers ready")

In [ ]:
# 実行セル。途中で止まっても history/global checkpoint があれば再利用します。
if not selected:
    raise RuntimeError("No selected variants")
if not SCOLQ_MODEL.exists():
    raise FileNotFoundError(f"Missing SCoLQ model: {SCOLQ_MODEL}")

for variant in selected:
    history = history_rows(variant)
    completed_phase2 = sum(1 for row in history if int(row.get("phase", 0)) == 2)
    print("\n===", variant["name"], "===")
    print(variant["description"])
    print(f"completed_phase2: {completed_phase2}/{PHASE2_ROUNDS_PER_VARIANT}")
    if completed_phase2 >= PHASE2_ROUNDS_PER_VARIANT:
        print("already completed")
        continue

    pid_path = variant_pid_path(variant)
    existing_pid = read_pid(pid_path)
    if pid_state(existing_pid) == "running":
        print("already running pid:", existing_pid)
        continue

    runner_log = variant_runner_log(variant)
    runner_log.parent.mkdir(parents=True, exist_ok=True)
    cmd = train_cmd(variant)
    env = variant_env(variant)
    print("cmd:", " ".join(cmd))
    print("runner_log:", runner_log)
    print("train_log:", variant_train_log(variant))

    with runner_log.open("a", encoding="utf-8", buffering=1) as log:
        log.write("\n" + "=" * 100 + "\n")
        log.write(" ".join(cmd) + "\n")
        process = subprocess.Popen(
            cmd,
            cwd=REPO_ROOT,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        pid_path.write_text(str(process.pid), encoding="utf-8")
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            log.write(line)
        rc = process.wait()
        if rc != 0:
            raise RuntimeError(f"{variant['name']} failed with exit code {rc}. See {runner_log}")
        print("variant completed:", variant["name"])

In [ ]:
# 進捗確認セル
rows = []
for variant in selected:
    history = history_rows(variant)
    latest = Path(history[-1]["global"]) if history else variant_work_root(variant) / "global_checkpoints" / "round000_warmup.pt"
    rows.append({
        "variant": variant["name"],
        "pid": read_pid(variant_pid_path(variant)),
        "pid_state": pid_state(read_pid(variant_pid_path(variant))),
        "phase2": f"{sum(1 for row in history if int(row.get('phase', 0)) == 2)}/{PHASE2_ROUNDS_PER_VARIANT}",
        "latest": str(latest),
        "latest_exists": latest.exists(),
        "free_gib": round(shutil.disk_usage(variant_work_root(variant)).free / 1024**3, 2) if variant_work_root(variant).exists() else None,
    })

display(pd.DataFrame(rows))

for variant in selected:
    print("\n===", variant["name"], "runner tail ===")
    print("\n".join(tail_lines(variant_runner_log(variant), 24)))

In [ ]:
# early/final checkpoint 評価。Phase2 は early peak しやすいので r001/r002/r003/r010 を見る。
EVAL_WORKSPACE = variant_work_root(selected[0])
REPORT_DIR = EVAL_WORKSPACE / "validation_reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

checkpoints: list[tuple[str, Path]] = []
seed03 = SOURCE_WORK_ROOT / "global_checkpoints" / "phase1_round003_global.pt"
seed12 = SOURCE_WORK_ROOT / "global_checkpoints" / "phase1_round012_global.pt"
old0834_c02 = DQA_ROOT / "efficientteacher_dqa08_3_4_phase2_feature_quality_sweep" / "c_feature_balanced_neck_head_r003" / "global_checkpoints" / "phase2_round002_global.pt"
old0834_d02 = DQA_ROOT / "efficientteacher_dqa08_3_4_phase2_feature_quality_sweep" / "d_feature_conservative_min_gate_r003" / "global_checkpoints" / "phase2_round002_global.pt"
checkpoints.extend([
    ("p1_r003", seed03),
    ("p1_r012", seed12),
    ("old08_3_4_c_r002", old0834_c02),
    ("old08_3_4_d_r002", old0834_d02),
])

for variant in selected:
    root = variant_work_root(variant) / "global_checkpoints"
    for r in [1, 2, 3, PHASE2_ROUNDS_PER_VARIANT]:
        ckpt = root / f"phase2_round{r:03d}_global.pt"
        if ckpt.exists():
            checkpoints.append((f"{variant['name']}_r{r:03d}", ckpt))

print("eval_workspace:", EVAL_WORKSPACE)
for label, path in checkpoints:
    print(label, path, "exists=", path.exists())

cmd = [
    sys.executable,
    str(EVAL_SCRIPT),
    "--workspace",
    str(EVAL_WORKSPACE),
    "--splits",
    "total",
    "--batch-size",
    "16",
    "--no-plots",
    "--verbose",
]
for label, path in checkpoints:
    if path.exists():
        cmd.extend(["--checkpoint", f"{label}={path}"])

print(" ".join(cmd))
subprocess.run(cmd, cwd=REPO_ROOT, check=True)

summary_csv = REPORT_DIR / "paper_protocol_eval_summary.csv"
if summary_csv.exists():
    df = pd.read_csv(summary_csv)
    out = REPORT_DIR / "paper_protocol_eval_summary_0836_early_total.csv"
    df.to_csv(out, index=False)
    display(df.sort_values(["split", "map50_95", "map50"], ascending=[True, False, False])[["checkpoint_label", "split", "precision", "recall", "map50", "map50_95"]])
    print("saved:", out)
else:
    print("No summary yet:", summary_csv)

In [ ]:
# 4 split final evaluation。上の total 評価で良い checkpoint を BEST_LABELS に入れる。
BEST_LABELS = []  # 例: ["a_scolq_soft_bbox_r003_r001", "c_scolq_classwise_weak_guard_r003_r002"]

ckpt_map = {label: path for label, path in checkpoints if path.exists()}
if not BEST_LABELS:
    BEST_LABELS = [
        label for label in ckpt_map
        if label.endswith("_r001") or label.endswith("_r002") or label.endswith(f"r{PHASE2_ROUNDS_PER_VARIANT:03d}")
    ]

cmd = [
    sys.executable,
    str(EVAL_SCRIPT),
    "--workspace",
    str(EVAL_WORKSPACE),
    "--splits",
    "highway,citystreet,residential,total",
    "--batch-size",
    "16",
    "--no-plots",
    "--verbose",
]
for label in ["p1_r003", "p1_r012", "old08_3_4_c_r002", "old08_3_4_d_r002", *BEST_LABELS]:
    path = ckpt_map.get(label)
    if path and path.exists():
        cmd.extend(["--checkpoint", f"{label}={path}"])

print(" ".join(cmd))
subprocess.run(cmd, cwd=REPO_ROOT, check=True)

summary_csv = REPORT_DIR / "paper_protocol_eval_summary.csv"
if summary_csv.exists():
    df = pd.read_csv(summary_csv)
    out = REPORT_DIR / "paper_protocol_eval_summary_0836_selected_4splits.csv"
    df.to_csv(out, index=False)
    display(df.sort_values(["split", "map50_95", "map50"], ascending=[True, False, False])[["checkpoint_label", "split", "precision", "recall", "map50", "map50_95"]])
    print("saved:", out)

In [ ]:
# pseudoGT / SCoLQ quality / DQA gate 推移を確認するセル。
rows = []
for variant in selected:
    stats_root = variant_stats_root(variant)
    for r in range(1, PHASE2_ROUNDS_PER_VARIANT + 1):
        path = stats_root / f"phase2_round{r:03d}.json"
        if not path.exists():
            continue
        data = json.loads(path.read_text())
        counts = [0.0] * 10
        qsum = [0.0] * 10
        confsum = [0.0] * 10
        objsum = [0.0] * 10
        for client in data.get("clients", []):
            for i, value in enumerate(client.get("counts", [])):
                counts[i] += float(value)
            for i, value in enumerate(client.get("quality_sums", [])):
                qsum[i] += float(value)
            for i, value in enumerate(client.get("confidence_sums", [])):
                confsum[i] += float(value)
            for i, value in enumerate(client.get("objectness_sums", [])):
                objsum[i] += float(value)
        total = sum(counts)
        rows.append({
            "variant": variant["name"],
            "round": r,
            "pseudo_total": total,
            "mean_scolq_quality": sum(qsum) / total if total else None,
            "mean_score_column": sum(confsum) / total if total else None,
            "mean_objectness": sum(objsum) / total if total else None,
            "active_classes": sum(1 for x in counts if x > 0),
            "person_count": counts[0],
            "car_count": counts[2],
            "traffic_light_count": counts[7],
            "traffic_sign_count": counts[8],
        })

stats_df = pd.DataFrame(rows)
if not stats_df.empty:
    display(stats_df)
    display(stats_df.groupby("variant")[["pseudo_total", "mean_scolq_quality", "mean_objectness", "active_classes"]].agg(["min", "max", "mean"]))
else:
    print("No pseudo stats yet")

for variant in selected:
    state_path = variant_work_root(variant) / "dqa_cwa_state.json"
    if not state_path.exists():
        continue
    state = json.loads(state_path.read_text())
    gate = state.get("phase2_scolq_policy", [])
    if gate:
        print()
        print(variant["name"])
        display(pd.DataFrame(gate))

## 読み方

A が `r001/r002` で Phase1 を超えるなら、SCoLQ bbox gate はそのまま採用候補。`r010` まで落ちる場合は、SCoLQ でも毎 round の target bbox regression が強すぎるので、B/C のように high gate を上げるか residual をさらに下げる。

C が A より安定するなら、問題は SCoLQ 全体ではなく class-wise に弱い bbox を流していること。D が良い場合は、Phase2 では bbox regression そのものをほぼ止め、SCoLQ は DQA aggregation と objectness repair のみに使う方がよい。